# Hybrid t=100 — one-shot, Drive-backed pipeline (v2, 2026-07-04)

**Everything in one run: build corpus -> train -> eval, all saved to Google Drive so a
runtime disconnect NEVER loses work.** Re-running from the top resumes: any stage whose
output already exists on Drive is skipped.

Drive layout (`MyDrive/qnm_t100/`):
- `dataset_sw_k4_t100.npz`, `dataset_sw_k2_t100.npz`  — the corpora (built once)
- `fno_sw_gate_s1em3_t100/`  — model_best.pt, history.json, eval/*.json

**To SKIP the ~2 h retrain** (you already trained once): upload the 3 files from your laptop
`outputs/hybrid/fno_sw_gate_s1em3_t100/` (model_best.pt, model_last.pt, history.json) into
`MyDrive/qnm_t100/fno_sw_gate_s1em3_t100/` before running Cell 6 — it will detect the model
and skip training. The corpus still rebuilds once (it was never saved; unavoidable).

**Rules:** run top-to-bottom; NEVER use File -> Save a copy in GitHub; the 96 GB GPU runs
batch 8 comfortably.

In [ ]:
# Cell 2 — mount Drive + work dir (persists across disconnects)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/qnm_t100'
os.makedirs(DRIVE, exist_ok=True)
print('Drive work dir:', DRIVE)
!ls -lh {DRIVE} 2>/dev/null || echo '(empty — first run)'

In [ ]:
# Cell 2b (OPTIONAL) — upload the already-trained model from your laptop -> Drive,
# to skip the ~2 h retrain. Run this AFTER Cell 2 (Drive mounted). A file picker opens:
# select model_best.pt, model_last.pt, history.json (ckpt.pt optional). Then Cell 6
# will detect the model on Drive and skip training. Skip this cell to train from scratch.
import os, shutil
from google.colab import files
DRIVE_OUT = '/content/drive/MyDrive/qnm_t100/fno_sw_gate_s1em3_t100'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Pick from your laptop: model_best.pt, model_last.pt, history.json')
up = files.upload()
for fn in list(up):
    shutil.move(fn, f'{DRIVE_OUT}/{fn}')
print('now on Drive:', sorted(os.listdir(DRIVE_OUT)))

In [ ]:
# Cell 3 — sanity: GPU / CPU / RAM
!nvidia-smi
!nproc
!free -g | head -2

In [ ]:
# Cell 4 — clone + pinned deps + allocator flag (set BEFORE torch import)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%cd /content
!rm -rf QNM_Project_Fresh
!git clone --depth 1 https://github.com/ScreaM835/QNM_Project_Fresh.git
REPO = '/content/QNM_Project_Fresh'
%cd {REPO}
!pip -q install "neuraloperator==2.0.0"
!mkdir -p outputs/hybrid
import torch; print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

In [ ]:
# Cell 5 — corpora: reuse from Drive if present, else build once and SAVE to Drive.
# k4 = prior (eval needs this); k2 = Richardson label (training needs this).
import os, shutil
%cd {REPO}
for k, name in [(4, 'dataset_sw_k4_t100.npz'), (2, 'dataset_sw_k2_t100.npz')]:
    drive_path = f'{DRIVE}/{name}'
    repo_path = f'{REPO}/outputs/hybrid/{name}'
    if os.path.exists(drive_path):
        print(f'[corpus] reuse {name} from Drive')
    else:
        print(f'[corpus] building k={k} (this is the slow one) ...')
        !python scripts/build_hybrid_dataset.py --config configs/hybrid_sw_dataset_t100.yaml --k {k} --out {repo_path}
        shutil.copy(repo_path, drive_path)
        print(f'[corpus] saved {name} -> Drive')
    if not os.path.exists(repo_path):
        os.symlink(drive_path, repo_path)   # link Drive copy into the repo
!ls -lh outputs/hybrid/*.npz

In [ ]:
# Cell 6 — train: skip if a model is already on Drive, else train and SAVE to Drive.
OUT = f'{REPO}/outputs/hybrid/fno_sw_gate_s1em3_t100'
DRIVE_OUT = f'{DRIVE}/fno_sw_gate_s1em3_t100'
if os.path.exists(f'{DRIVE_OUT}/model_best.pt'):
    print('[train] model found on Drive -> restoring, skipping training')
    shutil.copytree(DRIVE_OUT, OUT, dirs_exist_ok=True)
else:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train_hybrid_fno.py --config configs/hybrid_sw_gate_s1em3_t100.yaml
    os.makedirs(DRIVE_OUT, exist_ok=True)
    shutil.copytree(OUT, DRIVE_OUT, dirs_exist_ok=True)
    print('[train] saved model -> Drive')

In [ ]:
# Cell 7 — training summary
import json
with open(f'{OUT}/history.json') as f:
    h = json.load(f)
hist = h['history']
print('epochs       :', len(hist))
print('best val MSE :', h['best_val_mse'])
print('first / last :', hist[0]['val_mse'], '/', hist[-1]['val_mse'])

In [ ]:
# Cell 8 — QNM eval: t_end scan {50,65,80,100}; each table prints live. Saved to Drive.
%cd {REPO}
for TE in [50, 65, 80, 100]:
    print(f'\n================  t_end = {TE}  ================')
    !python scripts/eval_hybrid_protocol1.py --config configs/hybrid_sw_gate_s1em3_t100.yaml --t_end {TE} --procs 8
import glob
os.makedirs(f'{DRIVE_OUT}/eval', exist_ok=True)
for f in glob.glob(f'{OUT}/eval/*.json'):
    shutil.copy(f, f'{DRIVE_OUT}/eval/')
print('\n[eval] saved JSONs -> ' + DRIVE_OUT + '/eval')

In [ ]:
# Cell 9 — (optional) also grab the eval JSONs onto the laptop as a backup
!zip -j /content/hybrid_t100_eval.zip {OUT}/eval/protocol1_xq10_te*.json
from google.colab import files
files.download('/content/hybrid_t100_eval.zip')